# Customer Segmentation Analysis 🎯

This notebook demonstrates customer segmentation using RFM analysis and clustering algorithms to identify customer groups and optimize marketing strategies.

## Learning Objectives
- RFM (Recency, Frequency, Monetary) analysis
- Customer clustering with K-means
- Customer lifetime value calculation
- Marketing campaign optimization
- Business intelligence insights

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")

## 1. Data Generation and Loading

In [ ]:
# Generate sample e-commerce customer data
def generate_customer_data(n_customers=1000):
    """Generate realistic e-commerce customer data for analysis"""
    np.random.seed(42)
    
    # Customer IDs
    customer_ids = [f'CUST_{i:04d}' for i in range(1, n_customers + 1)]
    
    # Generate customer data
    data = []
    for cust_id in customer_ids:
        # Number of orders (1-20)
        n_orders = np.random.poisson(5) + 1
        
        # Total amount spent ($10-$2000)
        total_amount = np.random.gamma(2, 100) + 10
        
        # Days since last purchase (1-365)
        days_since_last = np.random.exponential(60) + 1
        
        # Average order value
        avg_order_value = total_amount / n_orders
        
        # Customer age (18-80)
        age = np.random.normal(45, 15)
        age = max(18, min(80, int(age)))
        
        # Gender
        gender = np.random.choice(['M', 'F'], p=[0.48, 0.52])
        
        # Location (simplified)
        location = np.random.choice(['Urban', 'Suburban', 'Rural'], p=[0.6, 0.3, 0.1])
        
        data.append({
            'customer_id': cust_id,
            'total_orders': n_orders,
            'total_amount': total_amount,
            'avg_order_value': avg_order_value,
            'days_since_last': days_since_last,
            'age': age,
            'gender': gender,
            'location': location
        })
    
    return pd.DataFrame(data)

# Generate customer data
customer_data = generate_customer_data(1000)
print(f"✅ Generated customer data: {customer_data.shape}")
print(f"📊 Sample data:")
print(customer_data.head())

## 2. RFM Analysis

In [ ]:
# Calculate RFM metrics
def calculate_rfm_scores(df):
    """Calculate RFM (Recency, Frequency, Monetary) scores"""
    # Create RFM dataframe
    rfm = df.copy()
    
    # Calculate RFM metrics
    rfm['recency'] = rfm['days_since_last']
    rfm['frequency'] = rfm['total_orders']
    rfm['monetary'] = rfm['total_amount']
    
    # Calculate RFM scores (1-5 scale, 5 being best)
    # Recency: Lower is better (more recent)
    rfm['r_score'] = pd.qcut(rfm['recency'], q=5, labels=[5, 4, 3, 2, 1])
    
    # Frequency: Higher is better
    rfm['f_score'] = pd.qcut(rfm['frequency'], q=5, labels=[1, 2, 3, 4, 5])
    
    # Monetary: Higher is better
    rfm['m_score'] = pd.qcut(rfm['monetary'], q=5, labels=[1, 2, 3, 4, 5])
    
    # Convert to numeric
    rfm['r_score'] = rfm['r_score'].astype(int)
    rfm['f_score'] = rfm['f_score'].astype(int)
    rfm['m_score'] = rfm['m_score'].astype(int)
    
    # Calculate RFM score (combination)
    rfm['rfm_score'] = rfm['r_score'] + rfm['f_score'] + rfm['m_score']
    
    # Create RFM segments
    def segment_customers(row):
        if row['rfm_score'] >= 13:
            return 'Champions'
        elif row['rfm_score'] >= 10:
            return 'Loyal Customers'
        elif row['rfm_score'] >= 8:
            return 'At Risk'
        elif row['rfm_score'] >= 6:
            return 'Can\'t Lose'
        else:
            return 'Lost'
    
    rfm['rfm_segment'] = rfm.apply(segment_customers, axis=1)
    
    return rfm

# Calculate RFM scores
rfm_data = calculate_rfm_scores(customer_data)
print("✅ RFM analysis completed!")
print(f"📊 RFM segments distribution:")
print(rfm_data['rfm_segment'].value_counts())

## 3. RFM Visualization

In [ ]:
# Create RFM analysis visualizations
def create_rfm_visualizations(rfm_data):
    """Create comprehensive RFM analysis visualizations"""
    # Create subplots
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('RFM Score Distribution', 'Segment Distribution', 
                       'Recency vs Frequency', 'Monetary vs Frequency'),
        specs=[[{"type": "histogram"}, {"type": "pie"}],
               [{"type": "scatter"}, {"type": "scatter"}]]
    )
    
    # RFM Score Distribution
    fig.add_trace(
        go.Histogram(x=rfm_data['rfm_score'], name='RFM Score', nbinsx=20),
        row=1, col=1
    )
    
    # Segment Distribution
    segment_counts = rfm_data['rfm_segment'].value_counts()
    fig.add_trace(
        go.Pie(labels=segment_counts.index, values=segment_counts.values, name='Segments'),
        row=1, col=2
    )
    
    # Recency vs Frequency
    fig.add_trace(
        go.Scatter(x=rfm_data['recency'], y=rfm_data['frequency'], 
                   mode='markers', name='Recency vs Frequency',
                   marker=dict(color=rfm_data['rfm_score'], colorscale='Viridis')),
        row=2, col=1
    )
    
    # Monetary vs Frequency
    fig.add_trace(
        go.Scatter(x=rfm_data['monetary'], y=rfm_data['frequency'], 
                   mode='markers', name='Monetary vs Frequency',
                   marker=dict(color=rfm_data['rfm_score'], colorscale='Viridis')),
        row=2, col=2
    )
    
    fig.update_layout(
        title='RFM Analysis Dashboard',
        height=800
    )
    
    return fig

# Create RFM visualizations
rfm_fig = create_rfm_visualizations(rfm_data)
rfm_fig.show()

## 4. K-Means Clustering

In [ ]:
# Prepare data for clustering
def prepare_clustering_data(rfm_data):
    """Prepare data for K-means clustering"""
    # Select features for clustering
    features = ['recency', 'frequency', 'monetary', 'avg_order_value', 'age']
    
    # Create feature matrix
    X = rfm_data[features].copy()
    
    # Handle missing values
    X = X.fillna(X.mean())
    
    # Standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    return X_scaled, features, scaler

# Find optimal number of clusters
def find_optimal_clusters(X, max_k=10):
    """Find optimal number of clusters using elbow method and silhouette score"""
    inertias = []
    silhouette_scores = []
    K_range = range(2, max_k + 1)
    
    for k in K_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(X)
        inertias.append(kmeans.inertia_)
        silhouette_scores.append(silhouette_score(X, kmeans.labels_))
    
    return K_range, inertias, silhouette_scores

# Prepare data and find optimal clusters
X_scaled, features, scaler = prepare_clustering_data(rfm_data)
K_range, inertias, silhouette_scores = find_optimal_clusters(X_scaled)

# Plot elbow curve and silhouette scores
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Elbow Method', 'Silhouette Score')
)

fig.add_trace(
    go.Scatter(x=list(K_range), y=inertias, mode='lines+markers', name='Inertia'),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=list(K_range), y=silhouette_scores, mode='lines+markers', name='Silhouette Score'),
    row=1, col=2
)

fig.update_layout(title='Optimal Number of Clusters', height=400)
fig.show()

# Choose optimal k (based on silhouette score)
optimal_k = K_range[np.argmax(silhouette_scores)]
print(f"🎯 Optimal number of clusters: {optimal_k}")

## 5. Customer Segmentation with K-Means

In [ ]:
# Perform K-means clustering
def perform_kmeans_clustering(X, n_clusters):
    """Perform K-means clustering and return results"""
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X)
    
    return kmeans, cluster_labels

# Perform clustering
kmeans_model, cluster_labels = perform_kmeans_clustering(X_scaled, optimal_k)

# Add cluster labels to data
rfm_data['cluster'] = cluster_labels

# Analyze clusters
def analyze_clusters(rfm_data, features):
    """Analyze cluster characteristics"""
    cluster_analysis = rfm_data.groupby('cluster')[features].mean()
    
    # Add cluster sizes
    cluster_sizes = rfm_data['cluster'].value_counts().sort_index()
    cluster_analysis['cluster_size'] = cluster_sizes.values
    cluster_analysis['cluster_percentage'] = (cluster_sizes.values / len(rfm_data)) * 100
    
    return cluster_analysis

# Analyze clusters
cluster_analysis = analyze_clusters(rfm_data, features)
print("📊 Cluster Analysis:")
print(cluster_analysis.round(2))

## 6. Cluster Visualization

In [ ]:
# Create cluster visualizations
def create_cluster_visualizations(rfm_data, features):
    """Create visualizations for cluster analysis"""
    # PCA for dimensionality reduction
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(rfm_data[features])
    
    # Create subplots
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Cluster Distribution (PCA)', 'Cluster Sizes', 
                       'Recency by Cluster', 'Monetary by Cluster'),
        specs=[[{"type": "scatter"}, {"type": "bar"}],
               [{"type": "box"}, {"type": "box"}]]
    )
    
    # PCA scatter plot
    for cluster in sorted(rfm_data['cluster'].unique()):
        mask = rfm_data['cluster'] == cluster
        fig.add_trace(
            go.Scatter(x=X_pca[mask, 0], y=X_pca[mask, 1], 
                       mode='markers', name=f'Cluster {cluster}',
                       marker=dict(size=8)),
            row=1, col=1
        )
    
    # Cluster sizes
    cluster_sizes = rfm_data['cluster'].value_counts().sort_index()
    fig.add_trace(
        go.Bar(x=cluster_sizes.index, y=cluster_sizes.values, name='Cluster Size'),
        row=1, col=2
    )
    
    # Recency by cluster
    for cluster in sorted(rfm_data['cluster'].unique()):
        mask = rfm_data['cluster'] == cluster
        fig.add_trace(
            go.Box(y=rfm_data[mask]['recency'], name=f'Cluster {cluster}'),
            row=2, col=1
        )
    
    # Monetary by cluster
    for cluster in sorted(rfm_data['cluster'].unique()):
        mask = rfm_data['cluster'] == cluster
        fig.add_trace(
            go.Box(y=rfm_data[mask]['monetary'], name=f'Cluster {cluster}'),
            row=2, col=2
        )
    
    fig.update_layout(
        title='Customer Segmentation Analysis',
        height=800
    )
    
    return fig

# Create cluster visualizations
cluster_fig = create_cluster_visualizations(rfm_data, features)
cluster_fig.show()

## 7. Customer Lifetime Value (CLV) Analysis

In [ ]:
# Calculate Customer Lifetime Value
def calculate_clv(rfm_data):
    """Calculate Customer Lifetime Value based on RFM metrics"""
    # Simple CLV calculation
    # CLV = (Average Order Value × Purchase Frequency × Customer Lifespan)
    
    # Estimate customer lifespan (in days) based on recency
    # Assuming customers with lower recency have longer lifespan
    max_recency = rfm_data['recency'].max()
    rfm_data['estimated_lifespan'] = max_recency - rfm_data['recency'] + 365
    
    # Calculate CLV
    rfm_data['clv'] = (
        rfm_data['avg_order_value'] * 
        rfm_data['frequency'] * 
        (rfm_data['estimated_lifespan'] / 365)  # Convert to years
    )
    
    return rfm_data

# Calculate CLV
rfm_data = calculate_clv(rfm_data)

# Analyze CLV by segments
clv_by_segment = rfm_data.groupby('rfm_segment')['clv'].agg(['mean', 'sum', 'count']).round(2)
clv_by_cluster = rfm_data.groupby('cluster')['clv'].agg(['mean', 'sum', 'count']).round(2)

print("💰 Customer Lifetime Value by RFM Segment:")
print(clv_by_segment)
print("\n💰 Customer Lifetime Value by Cluster:")
print(clv_by_cluster)

## 8. Marketing Recommendations

In [ ]:
# Generate marketing recommendations
def generate_marketing_recommendations(rfm_data):
    """Generate marketing recommendations based on segmentation"""
    recommendations = {}
    
    # RFM Segment recommendations
    rfm_recommendations = {
        'Champions': {
            'strategy': 'VIP treatment and loyalty programs',
            'actions': ['Exclusive early access to new products', 'Premium customer service', 'Referral rewards'],
            'budget_allocation': 'High'
        },
        'Loyal Customers': {
            'strategy': 'Cross-selling and upselling',
            'actions': ['Personalized recommendations', 'Bundle offers', 'Membership upgrades'],
            'budget_allocation': 'Medium-High'
        },
        'At Risk': {
            'strategy': 'Re-engagement campaigns',
            'actions': ['Win-back emails', 'Special discounts', 'Feedback surveys'],
            'budget_allocation': 'Medium'
        },
        "Can't Lose": {
            'strategy': 'Aggressive retention',
            'actions': ['Heavy discounts', 'Personal outreach', 'Product recommendations'],
            'budget_allocation': 'High'
        },
        'Lost': {
            'strategy': 'Re-acquisition campaigns',
            'actions': ['New customer offers', 'Social media campaigns', 'Influencer partnerships'],
            'budget_allocation': 'Low-Medium'
        }
    }
    
    # Cluster-based recommendations
    cluster_recommendations = {}
    for cluster in rfm_data['cluster'].unique():
        cluster_data = rfm_data[rfm_data['cluster'] == cluster]
        avg_clv = cluster_data['clv'].mean()
        avg_recency = cluster_data['recency'].mean()
        avg_frequency = cluster_data['frequency'].mean()
        
        if avg_clv > rfm_data['clv'].mean() and avg_recency < rfm_data['recency'].mean():
            strategy = 'Premium retention'
        elif avg_frequency > rfm_data['frequency'].mean():
            strategy = 'Frequency optimization'
        else:
            strategy = 'Value enhancement'
        
        cluster_recommendations[cluster] = {
            'strategy': strategy,
            'avg_clv': avg_clv,
            'avg_recency': avg_recency,
            'avg_frequency': avg_frequency
        }
    
    return rfm_recommendations, cluster_recommendations

# Generate recommendations
rfm_recs, cluster_recs = generate_marketing_recommendations(rfm_data)

print("🎯 Marketing Recommendations by RFM Segment:")
for segment, rec in rfm_recommendations.items():
    print(f"\n{segment}:")
    print(f"  Strategy: {rec['strategy']}")
    print(f"  Actions: {', '.join(rec['actions'])}")
    print(f"  Budget: {rec['budget_allocation']}")

print("\n🎯 Marketing Recommendations by Cluster:")
for cluster, rec in cluster_recs.items():
    print(f"\nCluster {cluster}:")
    print(f"  Strategy: {rec['strategy']}")
    print(f"  Avg CLV: ${rec['avg_clv']:.2f}")
    print(f"  Avg Recency: {rec['avg_recency']:.1f} days")
    print(f"  Avg Frequency: {rec['avg_frequency']:.1f} orders")

## 9. Key Insights and Conclusions

### 📊 **Key Findings:**
1. **Customer Segments**: Identified distinct customer groups with different behaviors and values
2. **RFM Analysis**: Champions and Loyal Customers represent the highest value segments
3. **Clustering**: K-means identified natural customer groups based on multiple features
4. **CLV Insights**: High-value customers have significantly higher lifetime value
5. **Marketing Opportunities**: Different segments require tailored marketing strategies

### 🎯 **Learning Outcomes:**
- ✅ RFM analysis and customer scoring
- ✅ K-means clustering for customer segmentation
- ✅ Customer lifetime value calculation
- ✅ Marketing strategy development
- ✅ Business intelligence and insights generation

### 🚀 **Next Steps:**
- Implement advanced clustering algorithms (DBSCAN, Hierarchical)
- Add predictive modeling for customer churn
- Create automated marketing campaign recommendations
- Build real-time customer segmentation dashboard
- Integrate with CRM systems for automated actions